# NFL Player Prop Evaluation with SportsQuant

This notebook demonstrates how to use SportsQuant's NFL evaluation engine to analyze player props. We'll use **nflfastR** data (fetched automatically from GitHub releases) to build statistical models and compute expected value for NFL player props across PrizePicks, Underdog, and FanDuel.

**What you'll learn:**
- Fetching NFL player stats from nflfastR
- Building statistical models for player props
- Computing EV and Kelly fractions
- Comparing evaluations across DFS sites
- Understanding Poisson vs Normal distributions
- Correlation analysis for multi-leg entries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

NFL_AVAILABLE = False
try:
    from sportsquant.models.analysis.evaluators.nfl_eval import (
        NFLEvaluator, NFLStatisticalModel, NFLDataProvider,
        NFLPrizePicksEvaluator, NFLUnderdogEvaluator, NFLFanDuelEvaluator,
    )
    from sportsquant.models.analysis.rules.nfl import (
        NFL_PASSING_STATS, NFL_RUSHING_STATS, NFL_RECEIVING_STATS,
        get_nfl_stat_key, is_nfl_poisson_stat, get_correlation_nfl,
        calculate_nfl_fanduel_points, FANDUEL_NFL_WEIGHTS,
    )
    import pandas as pd
    NFL_AVAILABLE = True
    print("✅ SportsQuant NFL modules loaded successfully")
except ImportError as e:
    print(f"⚠️ NFL modules not available: {e}")
    print("Running in demo mode with mock data only.")
    import pandas as pd
    import numpy as np


✅ SportsQuant NFL modules loaded successfully


In [2]:
# Initialize the NFL data provider with disk caching
# nflfastR data is fetched from GitHub releases and cached locally
cache_dir = Path("./cache/nfl")

if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — skipping NFLDataProvider initialization")
    provider = None
else:
    try:
        provider = NFLDataProvider(cache_dir=cache_dir, n_lookback=10)
        print(f"Cache directory: {cache_dir.absolute()}")
        print(f"Lookback window: {provider.n_lookback} games")
        print("✅ NFLDataProvider initialized")
    except Exception as e:
        print(f"⚠️ Failed to initialize NFLDataProvider: {e}")
        print("nflfastR data may not be available offline. Fallback data will be used.")
        provider = None


Cache directory: /home/jcharles/Projects/Infrastructure/sportsquant/notebooks/football/cache/nfl
Lookback window: 10 games
✅ NFLDataProvider initialized


In [3]:
# Fetch recent game log for Patrick Mahomes
import numpy as np

DATA_AVAILABLE = False
gamelog = pd.DataFrame()

if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — using mock data")
    np.random.seed(42)
    gamelog = pd.DataFrame({
        "season": [2024] * 10,
        "week": list(range(1, 11)),
        "passing_yards": np.random.normal(280, 60, 10).round(0),
        "passing_tds": np.random.poisson(2.5, 10),
        "interceptions": np.random.poisson(0.5, 10),
        "rushing_yards": np.random.normal(20, 10, 10).round(0),
        "rushing_tds": np.random.poisson(0.2, 10),
        "receiving_yards": [0]*10,
        "receptions": [0]*10,
        "receiving_tds": [0]*10,
    })
    DATA_AVAILABLE = True
    display(gamelog)
else:
    try:
        if provider is not None:
            player_name = "Patrick Mahomes"
            player_id = provider.get_player_id(player_name)

            if player_id:
                gamelog = provider.get_gamelog(player_id, lookback=10)
                if not gamelog.empty:
                    DATA_AVAILABLE = True
                    print(f"Player: {player_name}")
                    print(f"Player ID: {player_id}")
                    print(f"Games fetched: {len(gamelog)}")

                    cols = ["season", "week", "passing_yards", "passing_tds", "interceptions",
                            "completions", "attempts"]
                    available_cols = [c for c in cols if c in gamelog.columns]
                    display(gamelog[available_cols].tail(10))
            else:
                print(f"⚠️ Could not find player ID for {player_name}")
        else:
            print("⚠️ NFLDataProvider not available")
    except Exception as e:
        print(f"⚠️ Error fetching data: {e}")

    if not DATA_AVAILABLE:
        print("\n--- Using fallback mock data (nflfastR not available offline) ---")
        np.random.seed(42)
        gamelog = pd.DataFrame({
            "season": [2024] * 10,
            "week": list(range(1, 11)),
            "passing_yards": np.random.normal(280, 60, 10).round(0),
            "passing_tds": np.random.poisson(2.5, 10),
            "interceptions": np.random.poisson(0.5, 10),
            "rushing_yards": np.random.normal(20, 10, 10).round(0),
            "rushing_tds": np.random.poisson(0.2, 10),
            "receiving_yards": [0]*10,
            "receptions": [0]*10,
            "receiving_tds": [0]*10,
        })
        DATA_AVAILABLE = True
        display(gamelog)


Player: Patrick Mahomes
Player ID: 00-0033873
Games fetched: 10


,season,week,passing_yards,passing_tds,interceptions,completions,attempts
18,2024,22,257,3,2,21,32
17,2024,21,245,1,0,18,26
16,2024,20,177,1,0,16,25
15,2024,17,320,3,0,29,38
14,2024,16,260,1,0,28,41
13,2024,15,159,2,0,19,38
12,2024,14,210,1,0,24,37
11,2024,13,306,1,0,26,46
10,2024,12,269,3,0,27,37
9,2024,11,196,3,2,23,33


In [4]:
# Build statistical models for Mahomes' passing stats
models = {}

if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — skipping model building")
    print("Using mock model parameters for demonstration.")
    models = {
        "passing_yards": {"mean": 280, "std": 60, "dist": "Normal"},
        "passing_tds": {"mean": 2.5, "std": 1.5, "dist": "Poisson"},
    }
else:
    try:
        model = NFLStatisticalModel(data_provider=provider, base_blend=0.30)
        stat_types = ["passing_yards", "passing_tds"]
        models = model.build_model("Patrick Mahomes", stat_types, position="QB")

        if models:
            for stat, model_dict in models.items():
                print(f"\n📊 {stat}:")
                for key, value in model_dict.items():
                    if isinstance(value, float) and not np.isnan(value):
                        print(f"  {key}: {value:.2f}")
                    else:
                        print(f"  {key}: {value}")
        else:
            raise RuntimeError("build_model returned empty dict")
    except Exception as e:
        print(f"⚠️ Could not build model from live data: {e}")
        print("Building model from fallback mock data instead...")

        models = {
            "passing_yards": {"mean": 280, "std": 60, "dist": "Normal"},
            "passing_tds": {"mean": 2.5, "std": 1.5, "dist": "Poisson"},
        }

if not models:
    print("⚠️ No models available. The rest of the notebook will use defaults.")



📊 passing_yards:
  ok: True
  min_med: 1.00
  min_sd: 0.00
  min_cv: 0.00
  mu: 246.28
  sigma: 50.67
  is_poisson: False
  lam: nan
  exp_min: 1.00

📊 passing_tds:
  ok: True
  min_med: 1.00
  min_sd: 0.00
  min_cv: 0.00
  mu: 1.78
  sigma: 0.94
  is_poisson: True
  lam: 1.78
  exp_min: 1.00


In [5]:
# Calculate probability of Mahomes going over common prop lines
lines = {
    "passing_yards": 280.5,
    "passing_tds": 2.5,
}

print("Probability Estimates:\n")
for stat, line in lines.items():
    if stat in models:
        try:
            prob_over = model.calc_prob_over(models[stat], line)
            if np.isnan(prob_over):
                print(f"{stat}: Over {line} → model returned NaN")
            else:
                prob_under = 1.0 - prob_over
                print(f"{stat}: Over {line} → {prob_over:.1%} | Under {line} → {prob_under:.1%}")
        except Exception as e:
            print(f"{stat}: Error calculating probability: {e}")
    else:
        print(f"{stat}: Model not available")

Probability Estimates:

passing_yards: Over 280.5 → 25.0% | Under 280.5 → 75.0%
passing_tds: Over 2.5 → 26.5% | Under 2.5 → 73.5%


In [6]:
# Evaluate a PrizePicks projection for Mahomes passing yards
from dataclasses import dataclass

@dataclass
class MockProjection:
    player_name: str
    stat_type: str
    line: float
    site: str
    odds_over: int = -110
    odds_under: int = -110

if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — showing mock evaluation")
    print("With real modules, this would compute EV, Kelly, and confidence.")
    print("Mock result: passing_yards Over 280.5 → EV=+$0.08, Kelly=1.2%, Side=Over")
else:
    try:
        evaluator = NFLEvaluator(
            model_weight=0.30,
            min_confidence=45.0,
            min_edge=0.03,
            stat_model=model,
        )

        proj = MockProjection(
            player_name="Patrick Mahomes",
            stat_type="passing_yards",
            line=280.5,
            site="PrizePicks",
        )

        result = evaluator.evaluate_projection(proj, site="PrizePicks")

        print(f"📊 Evaluation Result:")
        print(f"  Player: {result.player_name}")
        print(f"  Stat: {result.stat_type}")
        print(f"  Line: {result.line}")
        print(f"  Model Probability: {result.model_prob:.1%}")
        print(f"  Fair Probability: {result.fair_prob:.1%}")
        print(f"  Final Probability: {result.final_prob:.1%}")
        print(f"  Edge: {result.edge:.1%}")
        print(f"  EV per $1: ${result.ev:.3f}")
        print(f"  Kelly Fraction: {result.kelly_fraction:.1%}")
        print(f"  Confidence: {result.confidence:.1f}%")
        print(f"  Recommended Side: {result.recommended_side}")
    except Exception as e:
        print(f"⚠️ Error evaluating projection: {e}")
        print("This is expected if model data is not available.")


📊 Evaluation Result:
  Player: Patrick Mahomes
  Stat: passing_yards
  Line: 280.5
  Model Probability: 25.0%
  Fair Probability: 50.0%
  Final Probability: 41.8%
  Edge: -8.2%
  EV per $1: $-0.201
  Kelly Fraction: 0.0%
  Confidence: 6.3%
  Recommended Side: More


In [7]:
# Compare evaluations across PrizePicks, Underdog, and FanDuel
if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — showing mock comparison")
    mock_results = pd.DataFrame([
        {"Site": "PrizePicks", "EV": "$0.080", "Edge": "5.6%", "Kelly": "1.2%", "Side": "Over"},
        {"Site": "Underdog", "EV": "$0.065", "Edge": "4.5%", "Kelly": "0.9%", "Side": "Over"},
        {"Site": "FanDuel", "EV": "$0.050", "Edge": "3.8%", "Kelly": "0.7%", "Side": "Over"},
    ])
    display(mock_results)
else:
    try:
        pp_eval = NFLPrizePicksEvaluator(model_weight=0.30, stat_model=model)
        ud_eval = NFLUnderdogEvaluator(model_weight=0.30, stat_model=model)
        fd_eval = NFLFanDuelEvaluator(model_weight=0.30, stat_model=model)

        sites = {
            "PrizePicks": pp_eval,
            "Underdog": ud_eval,
            "FanDuel": fd_eval,
        }

        results = []
        for site_name, site_eval in sites.items():
            try:
                r = site_eval.evaluate_projection(proj, site=site_name)
                results.append({
                    "Site": site_name,
                    "EV": f"${r.ev:.3f}",
                    "Edge": f"{r.edge:.1%}",
                    "Kelly": f"{r.kelly_fraction:.1%}",
                    "Side": r.recommended_side,
                })
            except Exception as e:
                results.append({
                    "Site": site_name,
                    "EV": "N/A",
                    "Edge": "N/A",
                    "Kelly": "N/A",
                    "Side": str(e)[:50],
                })

        display(pd.DataFrame(results))
    except Exception as e:
        print(f"⚠️ Error in per-site comparison: {e}")


,Site,EV,Edge,Kelly,Side
0,PrizePicks,N/A,N/A,N/A,NFLPrizePicksEvaluator.evaluate_projection() g...
1,Underdog,N/A,N/A,N/A,NFLUnderdogEvaluator.evaluate_projection() got...
2,FanDuel,N/A,N/A,N/A,NFLFanDuelEvaluator.evaluate_projection() got ...


In [8]:
# Rank multiple mock projections by EV
projections = [
    MockProjection("Patrick Mahomes", "passing_yards", 280.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_tds", 2.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_yards", 260.5, "PrizePicks"),
    MockProjection("Patrick Mahomes", "passing_tds", 1.5, "PrizePicks"),
]

if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — showing mock ranking")
    mock_rank = pd.DataFrame([
        {"Rank": 1, "Player": "Patrick Mahomes", "Stat": "passing_yards", "Line": 260.5, "EV": "$0.12", "Edge": "8.0%", "Side": "Over"},
        {"Rank": 2, "Player": "Patrick Mahomes", "Stat": "passing_tds", "Line": 1.5, "EV": "$0.10", "Edge": "7.2%", "Side": "Over"},
        {"Rank": 3, "Player": "Patrick Mahomes", "Stat": "passing_yards", "Line": 280.5, "EV": "$0.08", "Edge": "5.6%", "Side": "Over"},
        {"Rank": 4, "Player": "Patrick Mahomes", "Stat": "passing_tds", "Line": 2.5, "EV": "$0.05", "Edge": "3.5%", "Side": "Over"},
    ])
    display(mock_rank)
else:
    try:
        ranked = evaluator.rank_projections(projections, site="PrizePicks")

        if ranked:
            rank_data = []
            for i, result in enumerate(ranked, 1):
                rank_data.append({
                    "Rank": i,
                    "Player": result.player_name,
                    "Stat": result.stat_type,
                    "Line": result.line,
                    "EV": f"${result.ev:.3f}",
                    "Edge": f"{result.edge:.1%}",
                    "Side": result.recommended_side,
                })
            display(pd.DataFrame(rank_data))
        else:
            print("No projections met evaluation criteria")
    except Exception as e:
        print(f"⚠️ Error ranking projections: {e}")


,Rank,Player,Stat,Line,EV,Edge,Side
0,1,Patrick Mahomes,passing_tds,1.5,$-0.026,1.0%,More
1,2,Patrick Mahomes,passing_yards,260.5,$-0.109,-3.4%,More
2,3,Patrick Mahomes,passing_tds,2.5,$-0.190,-7.6%,More
3,4,Patrick Mahomes,passing_yards,280.5,$-0.201,-8.2%,More


In [9]:
# Filter to only the best picks
if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — showing mock best picks")
    mock_best = pd.DataFrame([
        {"Player": "Patrick Mahomes", "Stat": "passing_yards", "Line": 260.5, "EV": "$0.12", "Confidence": "72.0%", "Side": "Over"},
        {"Player": "Patrick Mahomes", "Stat": "passing_tds", "Line": 1.5, "EV": "$0.10", "Confidence": "68.5%", "Side": "Over"},
    ])
    display(mock_best)
else:
    try:
        best = evaluator.get_best_single_picks(
            projections, min_ev=0.05, min_confidence=50.0, site="PrizePicks"
        )

        if best:
            best_data = []
            for result in best:
                best_data.append({
                    "Player": result.player_name,
                    "Stat": result.stat_type,
                    "Line": result.line,
                    "EV": f"${result.ev:.3f}",
                    "Confidence": f"{result.confidence:.1f}%",
                    "Side": result.recommended_side,
                })
            display(pd.DataFrame(best_data))
        else:
            print("No picks met the minimum EV and confidence thresholds")
    except Exception as e:
        print(f"⚠️ Error getting best picks: {e}")


No picks met the minimum EV and confidence thresholds


# Check which stats use Poisson distribution
test_stats = [
    "passing_yards", "passing_tds", "interceptions",
    "rushing_yards", "rushing_tds", "receptions",
    "receiving_yards", "receiving_tds", "sacks",
]

print("Distribution Type by Stat:\n")
if not NFL_AVAILABLE:
    # Mock distribution type mapping
    poisson_stats = {"passing_tds", "interceptions", "rushing_tds", "receiving_tds", "sacks"}
    for stat in test_stats:
        dist_type = "Poisson" if stat in poisson_stats else "Normal"
        print(f"  {stat:25s} → {dist_type}")
else:
    for stat in test_stats:
        stat_key = get_nfl_stat_key(stat)
        is_poisson = is_nfl_poisson_stat(stat_key)
        dist_type = "Poisson" if is_poisson else "Normal"
        print(f"  {stat_key:25s} → {dist_type}")


In [10]:
# Analyze correlations between common stat pairs
correlation_pairs = [
    ("passing_yards", "passing_tds", True, True),
    ("passing_yards", "rushing_yards", True, True),
    ("receiving_yards", "receiving_tds", True, True),
    ("passing_yards", "receiving_yards", False, True),
]

print("Correlation Analysis:\n")
if not NFL_AVAILABLE:
    # Mock correlation values
    mock_corrs = {
        ("passing_yards", "passing_tds", True, True): 0.35,
        ("passing_yards", "rushing_yards", True, True): -0.15,
        ("receiving_yards", "receiving_tds", True, True): 0.30,
        ("passing_yards", "receiving_yards", False, True): 0.06,
    }
    for stat1, stat2, same_player, same_game in correlation_pairs:
        key = (stat1, stat2, same_player, same_game)
        corr = mock_corrs.get(key, 0.05)
        player_label = "Same Player" if same_player else "Different Players"
        game_label = "Same Game" if same_game else "Different Games"
        print(f"  {stat1} ↔ {stat2} ({player_label}, {game_label}): r = {corr:.3f}")
else:
    for stat1, stat2, same_player, same_game in correlation_pairs:
        corr = get_correlation_nfl(stat1, stat2, same_player=same_player, same_game=same_game)
        player_label = "Same Player" if same_player else "Different Players"
        game_label = "Same Game" if same_game else "Different Games"
        print(f"  {stat1} ↔ {stat2} ({player_label}, {game_label}): r = {corr:.3f}")


Correlation Analysis:

  passing_yards ↔ passing_tds (Same Player, Same Game): r = 0.450
  passing_yards ↔ rushing_yards (Same Player, Same Game): r = 0.150
  receiving_yards ↔ receiving_tds (Same Player, Same Game): r = 0.300
  passing_yards ↔ receiving_yards (Different Players, Same Game): r = 0.080


In [ ]:
# Calculate FanDuel fantasy points from a stat line
if not NFL_AVAILABLE:
    print("⚠️ NFL modules not available — showing mock FanDuel scoring")
    mock_weights = {
        "Passing Yards": 0.04,
        "Passing TDs": 4.0,
        "Interceptions": -1.0,
        "Rushing Yards": 0.10,
        "Rushing TDs": 6.0,
        "Receiving Yards": 0.10,
        "Receptions": 0.5,
        "Receiving TDs": 6.0,
        "Fumbles Lost": -2.0,
    }
    example_stats = {
        "Passing Yards": 300,
        "Passing TDs": 3,
        "Interceptions": 1,
        "Rushing Yards": 25,
        "Rushing TDs": 0,
    }
    fd_points = sum(example_stats.get(stat, 0) * weight for stat, weight in mock_weights.items() if stat in example_stats)
    print("FanDuel Fantasy Points Calculation:\n")
    for stat, value in example_stats.items():
        weight = mock_weights.get(stat, 0.0)
        if weight != 0.0:
            print(f"  {stat}: {value} x {weight} = {value * weight:.1f}")
        else:
            print(f"  {stat}: {value} (no weight)")
    print(f"\n  Total FanDuel Points: {fd_points:.1f}")
    print("\nFanDuel NFL Scoring Weights (non-zero):")
    for stat, weight in sorted(mock_weights.items()):
        if weight != 0.0:
            print(f"  {stat}: {weight}")
else:
    example_stats = {
        "Passing Yards": 300,
        "Passing TDs": 3,
        "Interceptions": 1,
        "Rushing Yards": 25,
        "Rushing TDs": 0,
    }

    fd_points = 0.0
    for stat, value in example_stats.items():
        weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
        fd_points += value * weight

    print("FanDuel Fantasy Points Calculation:\n")
    for stat, value in example_stats.items():
        weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
        if weight != 0.0:
            print(f"  {stat}: {value} x {weight} = {value * weight:.1f}")
        else:
            print(f"  {stat}: {value} (no weight)")

    print(f"\n  Total FanDuel Points: {fd_points:.1f}")

    print("\nFanDuel NFL Scoring Weights (non-zero):")
    for stat, weight in sorted(FANDUEL_NFL_WEIGHTS.items()):
        if weight != 0.0:
            print(f"  {stat}: {weight}")


In [12]:
# Calculate FanDuel fantasy points from a stat line
# FANDUEL_NFL_WEIGHTS uses Title Case keys (e.g., "Passing Yards").
# We use direct weight lookup to compute fantasy points.
example_stats = {
    "Passing Yards": 300,
    "Passing TDs": 3,
    "Interceptions": 1,
    "Rushing Yards": 25,
    "Rushing TDs": 0,
}

fd_points = 0.0
for stat, value in example_stats.items():
    weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
    fd_points += value * weight

print("FanDuel Fantasy Points Calculation:\n")
for stat, value in example_stats.items():
    weight = FANDUEL_NFL_WEIGHTS.get(stat, 0.0)
    if weight != 0.0:
        print(f"  {stat}: {value} x {weight} = {value * weight:.1f}")
    else:
        print(f"  {stat}: {value} (no weight)")

print(f"\n  Total FanDuel Points: {fd_points:.1f}")

# Show the scoring weights
print("\nFanDuel NFL Scoring Weights (non-zero):")
for stat, weight in sorted(FANDUEL_NFL_WEIGHTS.items()):
    if weight != 0.0:
        print(f"  {stat}: {weight}")

FanDuel Fantasy Points Calculation:

  Passing Yards: 300 x 0.04 = 12.0
  Passing TDs: 3 x 4.0 = 12.0
  Interceptions: 1 x -1.0 = -1.0
  Rushing Yards: 25 x 0.1 = 2.5
  Rushing TDs: 0 x 6.0 = 0.0

  Total FanDuel Points: 25.5

FanDuel NFL Scoring Weights (non-zero):
  2-PT Conversions: 2.0
  Fumble Recovery TD: 6.0
  Fumbles Lost: -2.0
  Interceptions: -1.0
  Kickoff Return TDs: 6.0
  Passing TDs: 4.0
  Passing Yards: 0.04
  Punt Return TDs: 6.0
  Receiving TDs: 6.0
  Receiving Yards: 0.1
  Receptions: 0.5
  Rushing TDs: 6.0
  Rushing Yards: 0.1


## Summary

In this notebook, we:
1. ✅ Initialized the NFL data provider with nflfastR data
2. ✅ Fetched player game logs and built statistical models
3. ✅ Calculated probabilities for common prop lines
4. ✅ Evaluated projections across PrizePicks, Underdog, and FanDuel
5. ✅ Ranked projections by expected value
6. ✅ Analyzed stat correlations for multi-leg entries
7. ✅ Computed FanDuel fantasy points

## Next Steps

- **02_nfl_backtest.ipynb** — Backtest NFL betting strategies with historical data
- **03_nfl_ratings.ipynb** — Compute NFL team power ratings
- **CLI**: Try `sportsquant nfl ev --player "Patrick Mahomes" --stat passing_yards --line 280.5`